# Midterm · Part B notebook
**16:540:580:90 Quality Management · Fall 2026 · Keystone Creamery**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ri128-rutgers/quality-management-fall2026/blob/main/midterm_partB.ipynb) · also runs in Azure ML Studio or any Jupyter.

1. *File → Save a copy in Drive*, then *Runtime → Run all.* Inputs are already filled in from the exam page.
2. Change only the cells marked **TRY IT** and **YOUR CODE**.
3. Copy your results into the exam page, then submit this notebook (`.ipynb`, outputs visible) with your answer PDF.

Midterm rule: AI assistants may help with code **syntax only**; note any use in the disclosure box.

In [ ]:
# Setup: loads the course library and the Case 2 consortium data
import os, urllib.request
RAW = 'https://raw.githubusercontent.com/ri128-rutgers/quality-management-fall2026/main/'
for f in ('tvm.py', 'case2_programs.csv'):
    if not os.path.exists(f): urllib.request.urlretrieve(RAW + f, f)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from tvm import pw, irr, p_a, p_f
pd.options.display.float_format = '{:,.0f}'.format
print('ready')

---
## B1 · Where should the AI models run?
Own edge servers, or a 3-year cloud contract renewed once. Six years, after tax. Costs are negative.

In [ ]:
TAX, MARR, YEARS = 0.25, 0.09, 6
MACRS = [0.20, 0.32, 0.192, 0.1152, 0.1152, 0.0576]     # 5-year property (IRS Pub. 946)

def own_option():
    btcf = [-480_000.0] + [-130_000.0] * YEARS                # servers now; power, support, admin each year
    dep = [0] + [480_000 * r for r in MACRS]
    tax = [0] + [TAX * (btcf[y] - dep[y]) for y in range(1, YEARS + 1)]   # negative tax = tax saving
    return pd.DataFrame({'before tax': btcf, 'depreciation': dep, 'tax': tax,
                         'after tax': [b - t for b, t in zip(btcf, tax)]})

def cloud_option(renewal_change=0.0):
    # Price falls 7%/yr. Years 4-6 are a renewal: renewal_change moves them up or down from that path.
    pay = [275_000 * 0.93 ** (y - 1) * (1 + (renewal_change if y >= 4 else 0)) for y in range(1, 7)]
    btcf = [-25_000] + [-p for p in pay]                    # setup fee, then contract payments
    tax = [TAX * b for b in btcf]                           # expensed in the year paid
    return pd.DataFrame({'before tax': btcf, 'depreciation': 0.0, 'tax': tax,
                         'after tax': [b - t for b, t in zip(btcf, tax)]})

def compare(renewal_change=0.0):
    o, c = own_option(), cloud_option(renewal_change)
    return pd.DataFrame({'Own servers': [pw(o['before tax'], MARR), pw(o['after tax'], MARR)],
                         'Cloud contract': [pw(c['before tax'], MARR), pw(c['after tax'], MARR)]},
                        index=['PW before tax', 'PW after tax'])

print('OWN'); display(own_option()); print('CLOUD'); display(cloud_option())
base = compare(); display(base)
for row in base.index:
    print(f"{row}: cheaper option = {base.loc[row].idxmax()}  (by ${abs(base.loc[row].diff().iloc[-1]):,.0f})")

In [ ]:
# TRY IT (B1c): the renewal price for years 4-6 is not locked. 0.10 means 10% above the expected price path.
# Try a few values (0.02, 0.05, 0.10) and find roughly where the answer flips.
renewal_change = 0.10
r = compare(renewal_change)
display(r)
print(f'With a {renewal_change:.0%} renewal change, cheaper after tax:', r.loc['PW after tax'].idxmax())

---
## B2 · What will the predictive-maintenance program cost?
**Step 1:** estimate development effort with the two Case 2 CERs. **Step 2:** simulate eight years of cost.

In [ ]:
progs = pd.read_csv('case2_programs.csv')
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_predict

FEAT, LOGGED = ['sensors', 'enterprise_integrations', 'ml_models', 'data_readiness_1to5'], ['sensors', 'enterprise_integrations', 'ml_models']
def design(df, vendor):
    X = df[FEAT].astype(float).copy()
    X['in_house'] = (vendor == 'in-house').astype(int); X['platform'] = (vendor == 'platform').astype(int)
    return X
X = design(progs, progs['vendor_type']); y = progs['dev_integration_person_months']
Xl = X.copy(); Xl[LOGGED] = np.log(Xl[LOGGED])
cv = KFold(5, shuffle=True, random_state=0)
ols = LinearRegression().fit(Xl, np.log(y))
log_cv = cross_val_predict(LinearRegression(), Xl, np.log(y), cv=cv)
gbm = GradientBoostingRegressor(n_estimators=300, max_depth=2, learning_rate=0.05, random_state=0)
gbm_cv = cross_val_predict(gbm, X, y, cv=cv); gbm.fit(X, y)
sigma = float(np.std(np.log(y) - log_cv, ddof=1))

NEW = dict(sensors=150, enterprise_integrations=2, ml_models=2, data_readiness_1to5=3)
nx = design(pd.DataFrame([NEW]), pd.Series(['integrator'])); nxl = nx.copy(); nxl[LOGGED] = np.log(nxl[LOGGED])
pm_ols, pm_gbm = float(np.exp(ols.predict(nxl))[0]), float(gbm.predict(nx)[0])

err = lambda p: float(np.mean(np.abs(p - y) / y))
cer = pd.DataFrame({'cross-validated error (MAPE)': [err(np.exp(log_cv)), err(gbm_cv)],
                    'Keystone estimate (person-months)': [pm_ols, pm_gbm]},
                   index=['OLS (log-linear)', 'Gradient-boosted'])
display(cer.style.format({'cross-validated error (MAPE)': '{:.0%}', 'Keystone estimate (person-months)': '{:.1f}'}))
print('OLS in-sample R² (log scale):', round(ols.score(Xl, np.log(y)), 2))
print('Keystone inside the data range on every driver:',
      all(progs[f].min() <= v <= progs[f].max() for f, v in NEW.items()))

In [ ]:
# YOUR CODE (B2d): replace the body so the function returns one value per iteration:
# $300,000 with probability 0.25, otherwise $0. Use the rng it is given (e.g. rng.random(n)).
def risk_event(n, rng):
    return np.zeros(n)          # <- replace this line

In [ ]:
rng = np.random.default_rng(2026); n = 20_000
draws = {'effort (PM)': rng.lognormal(np.log(pm_ols), sigma, n),
         'labor rate ($/PM)': rng.triangular(15_000, 16_200, 18_500, n),
         'cloud ($/sensor/mo)': rng.triangular(24, 30, 38, n),
         'sustainment share': rng.triangular(0.11, 0.16, 0.21, n)}

def correlate(d, rho, seed):     # effort and labor rate rise together (Iman–Conover style reordering)
    z = np.random.default_rng(seed).multivariate_normal([0, 0], [[1, rho], [rho, 1]], n)
    out = dict(d)
    for j, k in enumerate(['effort (PM)', 'labor rate ($/PM)']):
        out[k] = np.sort(d[k])[stats.rankdata(z[:, j], method='ordinal').astype(int) - 1]
    return out

HW = 150 * 395 + 6 * 2_750; PA = p_a(0.08, 8)
def lcc(d, risk=0.0):
    dev = d['effort (PM)'] * d['labor rate ($/PM)']
    yearly = 150 * d['cloud ($/sensor/mo)'] * 12 + d['sustainment share'] * dev + 2 * 18_000
    return dev + HW + yearly * PA + 0.35 * HW * p_f(0.08, 4) + risk

corr = correlate(draws, 0.5, 2027)
risk = risk_event(n, np.random.default_rng(7))
point = float(lcc({'effort (PM)': pm_ols, 'labor rate ($/PM)': 16_200, 'cloud ($/sensor/mo)': 30, 'sustainment share': 0.16}))
runs = {'independent': lcc(draws), 'correlated': lcc(corr), 'correlated + risk event': lcc(corr, risk)}
summary = pd.DataFrame({k: {'P50': np.quantile(v, .5), 'P80': np.quantile(v, .8),
                            'chance point estimate holds': float((v <= point).mean())} for k, v in runs.items()}).T
print(f'Point estimate: ${point:,.0f}    risk event hit rate: {np.mean(risk > 0):.1%}')
display(summary.style.format({'P50': '${:,.0f}', 'P80': '${:,.0f}', 'chance point estimate holds': '{:.0%}'}))

for k, v in runs.items(): plt.plot(np.sort(v) / 1e6, np.linspace(0, 1, n), label=k)
plt.axvline(point / 1e6, color='k', ls='--', label='point estimate'); plt.axhline(0.8, color='grey', lw=.5)
plt.xlabel('8-year life-cycle cost ($M, discounted)'); plt.ylabel('probability cost is at or below')
plt.legend(); plt.title('S-curves'); plt.show()

tornado = pd.Series({k: stats.spearmanr(corr[k], runs['correlated'])[0] for k in corr}).sort_values()
tornado.plot.barh(); plt.title('What drives the spread?'); plt.xlabel('rank correlation with total cost'); plt.show()

**Check (B2d):** the hit rate printed above should be close to 25%. Say on the exam page how you checked your function.